In [1]:
# Cell 1: Setup
import os, gc, json, time
from pathlib import Path
import numpy as np, pandas as pd
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as exc:
    print('Không mount được Drive:', exc)

QUICK = False
QUICK_ROWS = 300_000
CHUNK = 1_000_000
GAP_HOURS = 2.0
TRAIN_FRAC, VAL_FRAC = 0.70, 0.10
ROOT = Path('/content/drive/MyDrive')
AUDIT = ROOT / 'THAT_IDS_RERUN' / 'journal_step19_dataset_audit_v1' / 'paper'
OUT = ROOT / 'THAT_IDS_RERUN' / 'journal_step19b_temporal_variants_v1' / ('quick' if QUICK else 'paper')
OUT.mkdir(parents=True, exist_ok=True)
DATASETS = {'UNSW_V3': ROOT / 'NF-UNSW-NB15-v3.csv', 'CIC_V3': ROOT / 'NF-CSE-CIC-IDS2018-v3.csv'}
COLS = json.load(open(AUDIT / 'columns.json'))   # feature lists from step19
LABEL_COL, ATTACK_COL = 'Label', 'Attack'
for k, p in DATASETS.items():
    print(k, 'OK' if p.exists() else 'MISSING', p, '| timestamps:', COLS[k]['timestamps'])

Mounted at /content/drive
UNSW_V3 OK /content/drive/MyDrive/NF-UNSW-NB15-v3.csv | timestamps: ['FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS']
CIC_V3 OK /content/drive/MyDrive/NF-CSE-CIC-IDS2018-v3.csv | timestamps: ['FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS']


In [2]:
# Cell 2: Read start time, label, family and the 47-feature hash
def vec_hash(df, cols):
    x = df[cols].apply(pd.to_numeric, errors='coerce').astype(np.float64)
    x = x.replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(np.float32)
    return pd.util.hash_pandas_object(x, index=False).to_numpy(np.uint64)

def load(name):
    cols = COLS[name]
    st_col = [c for c in cols['timestamps'] if 'START' in c.upper()][0]
    usecols = list(dict.fromkeys([LABEL_COL, ATTACK_COL, st_col] + cols['features_a']))
    parts = {'start': [], 'label': [], 'attack': [], 'ha': []}
    t0 = time.time(); n = 0
    for df in pd.read_csv(DATASETS[name], usecols=usecols, chunksize=CHUNK, low_memory=False,
                          nrows=QUICK_ROWS if QUICK else None):
        parts['start'].append(pd.to_numeric(df[st_col], errors='coerce').to_numpy(np.float64))
        parts['label'].append(pd.to_numeric(df[LABEL_COL], errors='coerce').fillna(0).astype(np.int8).to_numpy())
        parts['attack'].append(df[ATTACK_COL].astype(str).str.strip().to_numpy())
        parts['ha'].append(vec_hash(df, cols['features_a']))
        n += len(df); print('  %s: %s dòng (%.1f phút)' % (name, f'{n:,}', (time.time() - t0) / 60))
        del df; gc.collect()
    d = {k: np.concatenate(v) for k, v in parts.items()}
    codes, names = pd.factorize(pd.Series(d.pop('attack')), sort=True)
    d['family'] = codes.astype(np.int16); d['family_names'] = [str(x) for x in names]
    d['benign_code'] = int(np.argmax([((d['family'] == i) & (d['label'] == 0)).sum() for i in range(len(names))]))
    if (~np.isfinite(d['start'])).any():
        raise RuntimeError('%s có %d dòng thiếu FLOW_START' % (name, int((~np.isfinite(d['start'])).sum())))
    return d

DATA = {k: load(k) for k in DATASETS}

  UNSW_V3: 1,000,000 dòng (0.3 phút)
  UNSW_V3: 2,000,000 dòng (0.5 phút)
  UNSW_V3: 2,365,424 dòng (0.5 phút)
  CIC_V3: 1,000,000 dòng (0.2 phút)
  CIC_V3: 2,000,000 dòng (0.4 phút)
  CIC_V3: 3,000,000 dòng (0.6 phút)
  CIC_V3: 4,000,000 dòng (0.7 phút)
  CIC_V3: 5,000,000 dòng (0.9 phút)
  CIC_V3: 6,000,000 dòng (1.1 phút)
  CIC_V3: 7,000,000 dòng (1.3 phút)
  CIC_V3: 8,000,000 dòng (1.5 phút)
  CIC_V3: 9,000,000 dòng (1.7 phút)
  CIC_V3: 10,000,000 dòng (1.9 phút)
  CIC_V3: 11,000,000 dòng (2.0 phút)
  CIC_V3: 12,000,000 dòng (2.2 phút)
  CIC_V3: 13,000,000 dòng (2.4 phút)
  CIC_V3: 14,000,000 dòng (2.6 phút)
  CIC_V3: 15,000,000 dòng (2.8 phút)
  CIC_V3: 16,000,000 dòng (3.0 phút)
  CIC_V3: 17,000,000 dòng (3.2 phút)
  CIC_V3: 18,000,000 dòng (3.4 phút)
  CIC_V3: 19,000,000 dòng (3.7 phút)
  CIC_V3: 20,000,000 dòng (3.9 phút)
  CIC_V3: 20,115,529 dòng (3.9 phút)


In [3]:
# Cell 3: Time units (UTC days, gap-based sessions) and a per-unit table
def utc_day(start_ms):
    return (start_ms // 86_400_000).astype(np.int64)

def sessions(start_ms, gap_hours):
    order = np.argsort(start_ms, kind='mergesort')
    s = start_ms[order]
    new = np.r_[True, np.diff(s) > gap_hours * 3_600_000]
    sid_sorted = np.cumsum(new) - 1
    sid = np.empty(len(s), np.int64); sid[order] = sid_sorted
    return sid

unit_rows = []
for name, d in DATA.items():
    d['day'] = utc_day(d['start']); d['session'] = sessions(d['start'], GAP_HOURS)
    names = d['family_names']
    for kind in ['day', 'session']:
        for u in np.unique(d[kind]):
            m = d[kind] == u
            fams = sorted({names[i] for i in np.unique(d['family'][m & (d['label'] == 1)])})
            unit_rows.append({'dataset': name, 'unit_kind': kind, 'unit': int(u), 'rows': int(m.sum()),
                              'attack_rate': float(d['label'][m].mean()),
                              'start_utc': str(pd.to_datetime(d['start'][m].min(), unit='ms')),
                              'end_utc': str(pd.to_datetime(d['start'][m].max(), unit='ms')),
                              'attack_families': ', '.join(fams)})
units = pd.DataFrame(unit_rows)
units.to_csv(OUT / 'time_units.csv', index=False)
pd.set_option('display.max_colwidth', 160); pd.set_option('display.width', 250)
display(units)

,dataset,unit_kind,unit,rows,attack_rate,start_utc,end_utc,attack_families
0,UNSW_V3,day,16457,1095574,0.015939,2015-01-22 11:49:36.907000,2015-01-22 23:59:59.150000,"Analysis, Backdoor, DoS, Exploits, Fuzzers, Generic, Reconnaissance, Shellcode, Worms"
1,UNSW_V3,day,16458,39778,0.000000,2015-01-23 00:00:04.438000,2015-01-23 00:25:23.759000,
2,UNSW_V3,day,16484,1230072,0.089613,2015-02-18 00:23:27.658000,2015-02-18 12:29:24.927000,"Analysis, Backdoor, DoS, Exploits, Fuzzers, Generic, Reconnaissance, Shellcode, Worms"
3,UNSW_V3,session,0,1135352,0.015380,2015-01-22 11:49:36.907000,2015-01-23 00:25:23.759000,"Analysis, Backdoor, DoS, Exploits, Fuzzers, Generic, Reconnaissance, Shellcode, Worms"
4,UNSW_V3,session,1,1230072,0.089613,2015-02-18 00:23:27.658000,2015-02-18 12:29:24.927000,"Analysis, Backdoor, DoS, Exploits, Fuzzers, Generic, Reconnaissance, Shellcode, Worms"
5,CIC_V3,day,17576,2138838,0.268928,2018-02-14 12:28:07.705000,2018-02-14 23:59:58.824000,"FTP-BruteForce, SSH-Bruteforce"
6,CIC_V3,day,17577,1665321,0.058451,2018-02-15 00:00:00.176000,2018-02-15 23:59:06.573000,"DoS_attacks-GoldenEye, DoS_attacks-Slowloris"
7,CIC_V3,day,17578,1784129,0.115253,2018-02-16 12:37:31.856000,2018-02-16 22:03:51.966000,"DoS_attacks-Hulk, DoS_attacks-SlowHTTPTest"
8,CIC_V3,day,17582,1829859,0.157711,2018-02-20 12:28:05.595000,2018-02-20 23:53:07.588000,DDoS_attacks-LOIC-HTTP
9,CIC_V3,day,17583,2712803,0.381805,2018-02-21 00:01:06.903000,2018-02-21 23:59:46.925000,"DDOS_attack-HOIC, DDOS_attack-LOIC-UDP"


In [4]:
# Cell 4: The four split variants
def chrono_cut(idx, start):
    # Assign 0/1/2 to the rows in idx by their own chronological order (70/10/20 by rows).
    o = idx[np.argsort(start[idx], kind='mergesort')]
    n = len(o); a = int(n * TRAIN_FRAC); b = int(n * (TRAIN_FRAC + VAL_FRAC))
    return o[:a], o[a:b], o[b:]

def assign_by_groups(start, group_ids):
    out = np.empty(len(start), np.int8)
    for g in np.unique(group_ids):
        idx = np.flatnonzero(group_ids == g)
        tr, va, te = chrono_cut(idx, start)
        out[tr] = 0; out[va] = 1; out[te] = 2
    return out

def variants(d):
    n = len(d['start'])
    v = {'A_global': assign_by_groups(d['start'], np.zeros(n, np.int64)),
         'B_per_utc_day': assign_by_groups(d['start'], d['day']),
         'C_per_session': assign_by_groups(d['start'], d['session'])}
    # D: per UTC day and per family (benign is its own family)
    v['D_per_day_family'] = assign_by_groups(d['start'], d['day'] * 1000 + d['family'].astype(np.int64))
    return v

def describe(name, d, variant, a):
    y, fam, names = d['label'], d['family'], d['family_names']
    row = {'dataset': name, 'variant': variant}
    for s, lab in enumerate(['train', 'val', 'test']):
        m = a == s
        row[lab + '_rows'] = int(m.sum()); row[lab + '_attack_rate'] = float(y[m].mean()) if m.any() else float('nan')
    present = {lab: set(np.unique(fam[(a == s) & (y == 1)])) for s, lab in enumerate(['train', 'val', 'test'])}
    att = [i for i in range(len(names)) if i != d['benign_code']]
    for lab in ['train', 'val', 'test']:
        row['absent_from_' + lab] = ', '.join(names[i] for i in att if i not in present[lab])
    te_att = (a == 2) & (y == 1)
    row['test_attack_unseen_in_train_share'] = float((te_att & ~np.isin(fam, list(present['train']))).sum() / max(te_att.sum(), 1))
    row['test_families_absent_from_train'] = ', '.join(names[i] for i in sorted(present['test'] - present['train']))
    row['test_hash_a_seen_in_train_rate'] = float(np.isin(d['ha'][a == 2], np.unique(d['ha'][a == 0])).mean())
    row['test_starts_before_last_train_flow_rate'] = float((d['start'][a == 2] < d['start'][a == 0].max()).mean())
    r = row['test_attack_rate'] / row['train_attack_rate'] if row['train_attack_rate'] > 0 else float('inf')
    row['test_over_train_attack_rate'] = float(r)
    row['passes_rule'] = bool(row['test_families_absent_from_train'] == '' and 0.5 <= r <= 2.0)
    return row

rows = []
for name, d in DATA.items():
    for vname, a in variants(d).items():
        rows.append(describe(name, d, vname, a))
res = pd.DataFrame(rows)
res.to_csv(OUT / 'temporal_variants.csv', index=False)
display(res[['dataset', 'variant', 'train_attack_rate', 'val_attack_rate', 'test_attack_rate', 'test_over_train_attack_rate',
             'test_attack_unseen_in_train_share', 'test_families_absent_from_train', 'absent_from_val',
             'test_hash_a_seen_in_train_rate', 'test_starts_before_last_train_flow_rate', 'passes_rule']])

order = ['A_global', 'C_per_session', 'B_per_utc_day', 'D_per_day_family']
choice = {}
for name in DATA:
    ok = [v for v in order if bool(res[(res.dataset == name) & (res.variant == v)]['passes_rule'].iloc[0])]
    choice[name] = ok[0] if ok else 'NONE'
json.dump({'choice': choice, 'gap_hours': GAP_HOURS, 'quick': QUICK}, open(OUT / 'choice.json', 'w'), indent=1)
print('Biến thể được chọn theo quy tắc:', choice)

,dataset,variant,train_attack_rate,val_attack_rate,test_attack_rate,test_over_train_attack_rate,test_attack_unseen_in_train_share,test_families_absent_from_train,absent_from_val,test_hash_a_seen_in_train_rate,test_starts_before_last_train_flow_rate,passes_rule
0,UNSW_V3,A_global,0.039370,0.089252,0.087494,2.222336,0.0,,,0.516909,0.000000,False
1,UNSW_V3,B_per_utc_day,0.057856,0.045907,0.044468,0.768596,0.0,,,0.428022,0.479978,True
2,UNSW_V3,C_per_session,0.057856,0.045907,0.044468,0.768597,0.0,,,0.421156,0.479978,True
3,UNSW_V3,D_per_day_family,0.053980,0.053958,0.054008,1.000519,0.0,,,0.426720,0.481199,True
4,CIC_V3,A_global,0.156599,0.041699,0.077546,0.495188,1.0,"Bot, Infilteration","Bot, Brute_Force_-Web, Brute_Force_-XSS, DDOS_attack-HOIC, DDOS_attack-LOIC-UDP, DDoS_attacks-LOIC-HTTP, DoS_attacks-GoldenEye, DoS_attacks-Hulk, DoS_attack...",0.204827,0.000000,False
5,CIC_V3,B_per_utc_day,0.144819,0.182781,0.048235,0.333070,0.0,,"DDOS_attack-LOIC-UDP, DDoS_attacks-LOIC-HTTP, DoS_attacks-GoldenEye, DoS_attacks-Hulk, DoS_attacks-SlowHTTPTest, DoS_attacks-Slowloris, FTP-BruteForce",0.233673,0.909007,False
6,CIC_V3,C_per_session,0.144840,0.182758,0.048174,0.332602,0.0,,"DDOS_attack-LOIC-UDP, DDoS_attacks-LOIC-HTTP, DoS_attacks-GoldenEye, DoS_attacks-Hulk, DoS_attacks-SlowHTTPTest, DoS_attacks-Slowloris, FTP-BruteForce",0.233648,0.909008,False
7,CIC_V3,D_per_day_family,0.129298,0.129297,0.129300,1.000014,0.0,,,0.259791,0.909008,True


Biến thể được chọn theo quy tắc: {'UNSW_V3': 'C_per_session', 'CIC_V3': 'D_per_day_family'}
